In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# keras imports
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

In [15]:
# Load the dataset
data = pd.read_csv('train.csv')
data

,VWTI,SWTI,CWTI,EI,Class
0,2.263400,-4.4862,3.65580,-0.612510,0
1,3.271800,1.7837,2.11610,0.613340,0
2,-3.941100,-12.8792,13.05970,-3.312500,1
3,0.519500,-3.2633,3.08950,-0.984900,0
4,2.569800,-4.4076,5.98560,0.078002,0
...,...,...,...,...,...
1091,1.640600,3.5488,1.39640,-0.364240,0
1092,-0.048008,-1.6037,8.47560,0.755580,0
1093,2.942100,7.4101,-0.97709,-0.884060,0
1094,1.964700,6.9383,0.57722,0.663770,0


In [16]:
# Separate features and target
X = data[['VWTI', 'SWTI', 'CWTI', 'EI']]  # Adjust these columns to your dataset
y = data['Class']


# Further split training data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)


# Ensure target is in binary format (0 or 1 for binary classification)
# No need for one-hot encoding in binary classification
y_train = y_train.values
y_val = y_val.values


# Check shapes
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")



X_train shape: (876, 4), y_train shape: (876,)
X_val shape: (220, 4), y_val shape: (220,)


In [18]:
X_train.shape[1]

4

In [29]:
# Model
model = Sequential([

    # Separate Input Layer
    Input(shape=(X_train.shape[1],)),   # 4 input features

    # Hidden Layers
    Dense(8, activation='relu'),
    Dropout(0.2),  #Dropout to reduce overfitting

    Dense(4, activation='relu'),

    # Output Layer
    Dense(1, activation='sigmoid') ## Output layer for binary classification hence sigmoid, else softmax
])

# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Early stopping: # Add early stopping to prevent overfitting
early_stopping = EarlyStopping(
    monitor='val_loss', # Monitors validation loss
    patience=5, # Stops training after 5 epochs with no improvement
    restore_best_weights=True ## Restores the best model weights
)

# Model summary
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_7 (Dense)                 │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 4)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81 (324.00 B)

 Trainable params: 81 (324.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[early_stopping]  # Use early stopping callback
)

Epoch 1/15
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4635 - loss: 0.7513 - val_accuracy: 0.4682 - val_loss: 0.7298
Epoch 2/15
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5080 - loss: 0.7224 - val_accuracy: 0.5182 - val_loss: 0.7006
Epoch 3/15
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5308 - loss: 0.7002 - val_accuracy: 0.5727 - val_loss: 0.6783
Epoch 4/15
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6084 - loss: 0.6723 - val_accuracy: 0.6182 - val_loss: 0.6572
Epoch 5/15
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6393 - loss: 0.6563 - val_accuracy: 0.6455 - val_loss: 0.6359
Epoch 6/15
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7021 - loss: 0.6284 - val_accuracy: 0.7000 - val_loss: 0.6117
Epoch 7/15
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7306 - loss: 0.6017 - val_accuracy: 0.7500 - val_loss: 0.5833
Epoch 8/15
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7466 - loss: 0.5778 - val_accuracy: 0.7545 - val_loss

In [20]:
# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(X_val, y_val)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8682 - loss: 0.3340 
Test Loss: 0.3340, Test Accuracy: 0.8682


In [21]:
# You can also generate predictions
y_pred = (model.predict(X_val) > 0.5).astype(int)


print("Classification Report:")
print(classification_report(y_val, y_pred))

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.81      0.88       135
           1       0.76      0.95      0.85        85

    accuracy                           0.87       220
   macro avg       0.86      0.88      0.87       220
weighted avg       0.89      0.87      0.87       220



In [22]:
import numpy as np


# Define the make_prediction function
def make_prediction(input_data):
    # Preprocess input data (apply scaling)
    # Use the pre-loaded scaler instead of creating a new one
    input_data_scaled = scaler.transform(input_data)  # Use transform instead of fit_transform

    # Use the trained model to predict the class
    predictions = model.predict(input_data_scaled)

    # Convert prediction to binary (0 or 1)
    predicted_classes = (predictions > 0.5).astype(int)

    # Return the prediction as a string
    if predicted_classes[0] == 1:
        return "Real"
    else:
        return "Fake"


In [23]:

# Example input data for prediction (replace with actual form data or array)
input_data = np.array([[1.5, 2.3, 3.4, 0.7]])  # Example data

# Get the prediction
result = make_prediction(input_data)
print(result)  # Will print "Real" or "Fake" based on the prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Fake


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [24]:

# Example input data for prediction (replace with actual form data or array)
input_data = np.array([[-3.9411, -12.8792,  13.0597,  -3.3125]])  # Example data

# Get the prediction
result = make_prediction(input_data)
print(result)  # Will print "Real" or "Fake" based on the prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Real


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [25]:
import pickle

model.save('models/model.h5')
# Save the scaler using pickle
with open('models/scaler.pkl', 'wb') as scaler_file:
    pickle.dump(scaler, scaler_file)

In [26]:
from tensorflow.keras.models import load_model

# Load the model
model = load_model('models/model.h5')
scaler = pickle.load(open("/content/models/scaler.pkl", "rb"))

In [27]:
def make_prediction(input_data):
    # Preprocess input data (apply scaling)
    input_data_scaled = scaler.transform(input_data)  # Use transform instead of fit_transform

    # Use the trained model to predict the class
    predictions = model.predict(input_data_scaled)

    # Convert prediction to binary (0 or 1)
    predicted_classes = (predictions > 0.5).astype(int)

    return predicted_classes

In [28]:
# Get form data
VWTI = 1.5
SWTI = 2.3
CWTI = 3.1
EI = 0.5

# Prepare input data for prediction
input_data = np.array([[VWTI, SWTI, CWTI, EI]])

# Get the prediction
result = make_prediction(input_data)
print(result)
if result[0] == 1:
  output = "real"
else:
  output = "fake"
print(output)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
[[0]]
fake


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
